# Add # slots: when missing, requires # intent:
In `process_conll_file('[your-file]', 'output.conll')`, replace `[your-file]` with your input path. Use a raw string if the path contains backslashes. The output file `output.conll` will contain `# slots:` lines.


In [ ]:
import re


with open('[your-file]', 'r', encoding='utf-8') as file:
    file_content = file.read()

# Replace lines that still contain English source text
updated_content = re.sub(r'(?m)^# text: (.*)', r'# text EN: \1', file_content)


with open('`[your-file]`', 'w', encoding='utf-8') as file:
    file.write(updated_content)

In [ ]:
def add_annotator_line(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    new_lines = []
    for line in lines:
        if line.startswith("# text EN:"):
            new_lines.append("# annotator: [your-name]\n") # set annotator name to whoever labeled the file
        new_lines.append(line)

    with open(filename, 'w', encoding='utf-8') as file:
        file.writelines(new_lines)

add_annotator_line('[your-file]')


In [ ]:
def process_conll_file(input_filename, output_filename):
    with open(input_filename, "r") as f:
        read = f.read()
    lines = read.split('\n')
    output_lines = []
    found_intent = False  # track whether intent line was seen

    # iterate over lines
    for i, line in enumerate(lines):
        # append current line to output
        output_lines.append(line)

        # check for intent header
        if line.startswith('# intent:'):
            # insert # slots: after intent when missing
            if (i + 1) < len(lines) and not lines[i + 1].startswith('# slots:'):
                # add # slots: line
                output_lines.append('# slots:')

        # reset flag after slots header
        if line.startswith('# slots:'):
            found_intent = False

    # join output lines
    output_text = '\n'.join(output_lines)

    # write updated text to output file
    with open(output_filename, "a") as f:
        f.write(output_text)

# set your input and output files
process_conll_file('[your-input-file]', '[your-output-file]')


# Ensure tab separators; later steps split lines on tabs to read slots



In [ ]:
# open file for in-place rewrite
with open('[your-file]', 'r+', encoding='utf-8') as f:
    # read file contents
    content = f.read()
    # split utterances on blank lines
    paragraphs = content.strip().split('\n\n')

    # store updated utterance blocks
    updated_paragraphs = []

    # process each utterance block
    for paragraph in paragraphs:
        # split block into lines
        lines = paragraph.strip().split('\n')
        # find # slots: line index
        slots_line_index = -1
        for i, line in enumerate(lines):
            if line.startswith('# slots:'):
                slots_line_index = i
                break

        # when slots header exists
        if slots_line_index != -1:
            # normalize token lines after slots header
            for j in range(slots_line_index + 1, len(lines)):
                line = lines[j]
                # split on whitespace
                elements = line.split()
                # rejoin tokens with tabs when spaces were used
                if len(elements) > 1:
                    # join fields with tab separators
                    tabbed_line = '\t'.join(elements)
                    # replace line with tab-separated version
                    lines[j] = tabbed_line

        # append updated block
        updated_paragraphs.append('\n'.join(lines))

    # rewrite file with updated blocks
    f.seek(0)
    f.write('\n\n'.join(updated_paragraphs))
    f.truncate()  # Обрезаем остаток содержимого файла, если он есть


# Fill in # slots. Two versions below: first for Tatar, then for Russian.
Set your input file in `with open('<your-file>', 'r+') as f:`

# Tatar version (Russian version is below)

In [ ]:
# open file for in-place rewrite
with open('[your-file]', 'r+') as f:
    # read file contents
    read = f.read()
    # split utterances on blank lines
    paragraphs = read.strip().split('\n\n')

    # store updated utterance blocks
    updated_paragraphs = []

    # process each utterance block
    for paragraph in paragraphs:
        # initialize list to store slots
        slots = []
        # split block into lines
        lines = paragraph.strip().split('\n')
        # get TAT text from the block
        tat_line = [line for line in lines if line.startswith('# text TAT:')][0]
        tat_text = tat_line.split(':', 1)[1].strip()  # extract text after '# text TAT:'

        # initial values for tracking the current slot
        current_slot_start = None
        current_slot_end = None
        current_mark = None

        # iterate over lines in the block
        for line in lines:
            # skip lines that don't contain word data
            if line.startswith('#'):
                continue

            # split line into elements: index, word, intent, mark
            parts = line.split('\t')
            if len(parts) != 4:
                continue

            index, word, intent, mark = parts
            mark = mark.strip()

            # check the word's tag
            if mark.startswith('O'):
                # if we hit 'O' and have an open slot, close it
                if current_slot_start is not None and current_slot_end is not None:
                    slots.append(f'{current_slot_start}:{current_slot_end}:{current_mark}')
                    current_slot_start = None
                    current_slot_end = None
                    current_mark = None
            else:
                # extract entity type from the tag (strip 'B-' or 'I-' prefix)
                entity_type = mark[2:]

                # find the word's start index in the TAT text
                word_index = tat_text.find(word)

                if mark.startswith('B-'):
                    # if the word starts with 'B-', start a new slot
                    if current_slot_start is not None and current_slot_end is not None:
                        slots.append(f'{current_slot_start}:{current_slot_end}:{current_mark}')
                    current_slot_start = word_index + 1  # offset index by 1 for the start
                    current_slot_end = word_index + len(word)
                    current_mark = entity_type
                elif mark.startswith('I-') and current_mark == entity_type:
                    # if the word starts with 'I-' and matches the current tag, extend the current slot
                    current_slot_end = word_index + len(word)

        # if there's an unfinished slot after processing the block, add it
        if current_slot_start is not None and current_slot_end is not None:
            slots.append(f'{current_slot_start}:{current_slot_end}:{current_mark}')

        # build the string for the # slots field
        slots_str = ', '.join(slots)

        # check whether a # slots line already exists in the block
        found_slots_line = False
        updated_lines = []
        for line in lines:
            if line.startswith('# slots:'):
                # if a # slots: line is found, replace it with the new slots_str
                updated_lines.append(f'# slots: {slots_str}')
                found_slots_line = True
            else:
                updated_lines.append(line)

        # if there was no # slots line, append it at the end of the block
        if not found_slots_line:
            updated_lines.append(f'# slots: {slots_str}')

        # append updated block
        updated_paragraphs.append('\n'.join(updated_lines))

    # rewrite file with updated blocks
    f.seek(0)
    f.write('\n\n'.join(updated_paragraphs))
    f.truncate()  # remove any leftover content in the file

# Russian version

In [ ]:
import logging

# logging configuration
logging.basicConfig(filename='error.log', level=logging.ERROR, format='%(asctime)s %(levelname)s: %(message)s')

# open file for in-place rewrite
with open('[your-file]', 'r+', encoding='utf-8') as f:
    # read file contents
    read = f.read()
    # split utterances on blank lines
    paragraphs = read.strip().split('\n\n')

    # store updated utterance blocks
    updated_paragraphs = []

    # process each utterance block
    for paragraph in paragraphs:
        try:
            # initialize list to store slots
            slots = []
            # split block into lines
            lines = paragraph.strip().split('\n')
            # get RU text from the block
            ru_line = [line for line in lines if line.startswith('# text RU:') or line.startswith('# test RU:')][0]
            ru_text = ru_line.split(':', 1)[1].strip()  # extract text after '# text RU:'

            # initial values for tracking the current slot
            current_slot_start = None
            current_slot_end = None
            current_mark = None

            # iterate over lines in the block
            for line in lines:
                # skip lines that don't contain word data
                if line.startswith('#'):
                    continue

                # split line into elements: index, word, intent, mark
                parts = line.split('\t')
                if len(parts) != 4:
                    continue

                index, word, intent, mark = parts
                mark = mark.strip()

                # check the word's tag
                if mark.startswith('O'):
                    # if we hit 'O' and have an open slot, close it
                    if current_slot_start is not None and current_slot_end is not None:
                        slots.append(f'{current_slot_start}:{current_slot_end}:{current_mark}')
                        current_slot_start = None
                        current_slot_end = None
                        current_mark = None
                else:
                    # extract entity type from the tag (strip 'B-' or 'I-' prefix)
                    entity_type = mark[2:]

                    # find the word's start index in the RU text
                    word_index = ru_text.find(word)

                    if mark.startswith('B-'):
                        # if the word starts with 'B-', start a new slot
                        if current_slot_start is not None and current_slot_end is not None:
                            slots.append(f'{current_slot_start}:{current_slot_end}:{current_mark}')
                        current_slot_start = word_index + 1  # offset index by 1 for the start
                        current_slot_end = word_index + len(word)
                        current_mark = entity_type
                    elif mark.startswith('I-') and current_mark == entity_type:
                        # if the word starts with 'I-' and matches the current tag, extend the current slot
                        current_slot_end = word_index + len(word)

            # if there's an unfinished slot after processing the block, add it
            if current_slot_start is not None and current_slot_end is not None:
                slots.append(f'{current_slot_start}:{current_slot_end}:{current_mark}')

            # build the string for the # slots field
            slots_str = ', '.join(slots)

            # check whether a # slots line already exists in the block
            found_slots_line = False
            updated_lines = []
            for line in lines:
                if line.startswith('# slots:'):
                    # if a # slots: line is found, replace it with the new slots_str
                    updated_lines.append(f'# slots: {slots_str}')
                    found_slots_line = True
                else:
                    updated_lines.append(line)

            # if there was no # slots line, append it at the end of the block
            if not found_slots_line:
                updated_lines.append(f'# slots: {slots_str}')

            # append updated block
            updated_paragraphs.append('\n'.join(updated_lines))

        except IndexError as e:
            # log the error and the block that caused it, along with the error name
            logging.error(f'Error processing paragraph: {paragraph}\nError: {e}')
            continue

    # rewrite file with updated blocks
    f.seek(0)
    f.write('\n\n'.join(updated_paragraphs))
    f.truncate()  # remove any leftover content in the file

